In [5]:
from google.colab import files
files.upload()

Saving orders.json to orders (1).json


{'orders (1).json': b'{\r\n  "orders": [\r\n    {"order_id":1,"customer":"Rahul","product":"Laptop","amount":75000,"city":"Hyderabad"},\r\n    {"order_id":2,"customer":"Sneha","product":"Mouse","amount":1500,"city":"Bangalore"},\r\n    {"order_id":3,"customer":"Arjun","product":"Keyboard","amount":3000,"city":"Chennai"},\r\n    {"order_id":4,"customer":"Priya","product":"Laptop","amount":75000,"city":"Hyderabad"},\r\n    {"order_id":5,"customer":"Karan","product":"Monitor","amount":12000,"city":"Mumbai"},\r\n    {"order_id":6,"customer":"Rahul","product":"Mouse","amount":1000,"city":"Hyderabad"},\r\n    {"order_id":7,"customer":"Sneha","product":"Laptop","amount":75000,"city":"Bangalore"},\r\n    {"order_id":8,"customer":"Arjun","product":"Keyboard","amount":3000,"city":"Chennai"},\r\n    {"order_id":9,"customer":"Priya","product":"Mouse","amount":2000,"city":"Hyderabad"},\r\n    {"order_id":10,"customer":"Rahul","product":"Monitor","amount":12000,"city":"Hyderabad"}\r\n  ]\r\n}'}

In [6]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('orders').getOrCreate()

In [7]:
df_json = spark.read.option("multiline", "true").json("orders.json")
df_json.show(truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|orders                                                                                                                                                                                                                                                                                                                                                                              |
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [8]:
from pyspark.sql.functions import explode

df_orders = df_json.select(explode("orders").alias("order")).select("order.*")
df_orders.show()

+------+---------+--------+--------+--------+
|amount|     city|customer|order_id| product|
+------+---------+--------+--------+--------+
| 75000|Hyderabad|   Rahul|       1|  Laptop|
|  1500|Bangalore|   Sneha|       2|   Mouse|
|  3000|  Chennai|   Arjun|       3|Keyboard|
| 75000|Hyderabad|   Priya|       4|  Laptop|
| 12000|   Mumbai|   Karan|       5| Monitor|
|  1000|Hyderabad|   Rahul|       6|   Mouse|
| 75000|Bangalore|   Sneha|       7|  Laptop|
|  3000|  Chennai|   Arjun|       8|Keyboard|
|  2000|Hyderabad|   Priya|       9|   Mouse|
| 12000|Hyderabad|   Rahul|      10| Monitor|
+------+---------+--------+--------+--------+



In [9]:
from pyspark.sql.functions import explode,col

df_orders = df_json.select(explode("orders").alias("order")).select("order.*")
df_orders.show()

+------+---------+--------+--------+--------+
|amount|     city|customer|order_id| product|
+------+---------+--------+--------+--------+
| 75000|Hyderabad|   Rahul|       1|  Laptop|
|  1500|Bangalore|   Sneha|       2|   Mouse|
|  3000|  Chennai|   Arjun|       3|Keyboard|
| 75000|Hyderabad|   Priya|       4|  Laptop|
| 12000|   Mumbai|   Karan|       5| Monitor|
|  1000|Hyderabad|   Rahul|       6|   Mouse|
| 75000|Bangalore|   Sneha|       7|  Laptop|
|  3000|  Chennai|   Arjun|       8|Keyboard|
|  2000|Hyderabad|   Priya|       9|   Mouse|
| 12000|Hyderabad|   Rahul|      10| Monitor|
+------+---------+--------+--------+--------+



In [10]:
df_orders.show()

+------+---------+--------+--------+--------+
|amount|     city|customer|order_id| product|
+------+---------+--------+--------+--------+
| 75000|Hyderabad|   Rahul|       1|  Laptop|
|  1500|Bangalore|   Sneha|       2|   Mouse|
|  3000|  Chennai|   Arjun|       3|Keyboard|
| 75000|Hyderabad|   Priya|       4|  Laptop|
| 12000|   Mumbai|   Karan|       5| Monitor|
|  1000|Hyderabad|   Rahul|       6|   Mouse|
| 75000|Bangalore|   Sneha|       7|  Laptop|
|  3000|  Chennai|   Arjun|       8|Keyboard|
|  2000|Hyderabad|   Priya|       9|   Mouse|
| 12000|Hyderabad|   Rahul|      10| Monitor|
+------+---------+--------+--------+--------+



In [11]:
df_orders.count()

10

In [12]:
from pyspark.sql.functions import sum

df_orders.select(sum("amount")).show()

+-----------+
|sum(amount)|
+-----------+
|     259500|
+-----------+



In [13]:
df_orders.groupBy("customer") \
    .agg(sum("amount").alias("total_spent")) \
    .show()

+--------+-----------+
|customer|total_spent|
+--------+-----------+
|   Sneha|      76500|
|   Priya|      77000|
|   Rahul|      88000|
|   Arjun|       6000|
|   Karan|      12000|
+--------+-----------+



In [14]:
df_orders.groupBy("customer") \
    .agg(sum("amount").alias("total_spent")) \
    .orderBy(col("total_spent").desc()) \
    .show(1)

+--------+-----------+
|customer|total_spent|
+--------+-----------+
|   Rahul|      88000|
+--------+-----------+
only showing top 1 row


In [15]:
df_orders.groupBy("product") \
    .agg(sum("amount").alias("total_sales")) \
    .show()

+--------+-----------+
| product|total_sales|
+--------+-----------+
|  Laptop|     225000|
|   Mouse|       4500|
|Keyboard|       6000|
| Monitor|      24000|
+--------+-----------+



In [16]:
df_orders.filter(df_orders.city == "Hyderabad").show()

+------+---------+--------+--------+-------+
|amount|     city|customer|order_id|product|
+------+---------+--------+--------+-------+
| 75000|Hyderabad|   Rahul|       1| Laptop|
| 75000|Hyderabad|   Priya|       4| Laptop|
|  1000|Hyderabad|   Rahul|       6|  Mouse|
|  2000|Hyderabad|   Priya|       9|  Mouse|
| 12000|Hyderabad|   Rahul|      10|Monitor|
+------+---------+--------+--------+-------+



In [17]:
df_orders.filter(df_orders.amount > 10000).show()

+------+---------+--------+--------+-------+
|amount|     city|customer|order_id|product|
+------+---------+--------+--------+-------+
| 75000|Hyderabad|   Rahul|       1| Laptop|
| 75000|Hyderabad|   Priya|       4| Laptop|
| 12000|   Mumbai|   Karan|       5|Monitor|
| 75000|Bangalore|   Sneha|       7| Laptop|
| 12000|Hyderabad|   Rahul|      10|Monitor|
+------+---------+--------+--------+-------+



In [18]:
df_orders.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|  Chennai|    2|
|   Mumbai|    1|
|Hyderabad|    5|
+---------+-----+

